---
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 11 dimulai**.
> **Sifat tugas:** individu.

### Konteks / Skenario

Manajemen "Kopi Nusantara" (skenario yang sama dari Pertemuan 7 & 9) ingin membangun **dashboard pemantauan pesanan real-time** untuk mengetahui performa tiap cabang setiap saat, bukan hanya laporan bulanan seperti sebelumnya. Anda ditugaskan membangun purwarupa (*prototype*) pipeline streaming yang mensimulasikan kebutuhan ini.

### Instruksi Pengerjaan

Buat notebook baru **`Tugas10_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession` (jangan lupa `spark.conf.set("spark.sql.shuffle.partitions", "4")`), lalu kerjakan bagian **A sampai E** berikut:

---

**A. Menyiapkan Streaming Source** *(bobot 15%)*

Definisikan skema eksplisit (`StructType`) untuk data pesanan dengan kolom: `pesanan_id` (string), `nama_cabang` (string), `nama_menu` (string), `qty` (integer), `harga_satuan` (integer), `waktu_pesan` (timestamp). Buat folder simulasi streaming baru bernama `tugas10_stream`, lalu definisikan `readStream` dari folder tersebut.

**Jalanakan SparkSession**

In [12]:
# Persiapan ulang untuk latihan
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, avg, col
import os
import shutil
import time
import pandas as pd

spark = SparkSession.builder \
    .appName("Tugas10") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

# Buat Folder simulasi Streaming
folder_tugas = "tugas10_stream"
shutil.rmtree(folder_tugas, ignore_errors=True)
os.makedirs(folder_tugas, exist_ok=True)

skema_tugas = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True)
])
print("Siap untuk latihan. Folder:", folder_tugas)

df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_tugas) \
    .option("header", True) \
    .load(folder_tugas)

print("Streaming DataFrame berhasil didefinisikan (belum berjalan - baru 'rencana').")
print("df_stream.isStreaming:", df_stream.isStreaming)

Siap untuk latihan. Folder: tugas10_stream
Streaming DataFrame berhasil didefinisikan (belum berjalan - baru 'rencana').
df_stream.isStreaming: True


**B. Agregasi per Cabang** *(bobot 20%)*

Tambahkan kolom turunan `total_harga` (`qty x harga_satuan`) pada `df_stream`, lalu buat agregasi `groupBy("nama_cabang")` yang menghitung total `total_harga` dan jumlah pesanan. Jalankan sebagai streaming query dengan sink `memory`, `queryName="agregasi_cabang"`.

In [13]:
# Melakukan agregasi - sintaks IDENTIK dengan groupBy/agg pada DataFrame batch (Pertemuan 4-5)
# Tambahkan kolom turunan
df_stream_with_total = df_stream.withColumn("total_harga", col("qty") * col("harga_satuan"))

agregasi_stream = df_stream_with_total.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)
# Jalankan sebagai streaming query dengan sink memory
query = agregasi_stream.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Transformasi berhasil didefinisikan.")

Transformasi berhasil didefinisikan.


**C. Agregasi Berbasis Window Waktu** *(bobot 20%)*

Buat streaming query **kedua** (definisikan ulang `readStream` dari folder yang sama, ini diperbolehkan — satu sumber data dapat dibaca oleh lebih dari satu streaming query sekaligus) yang mengelompokkan total `total_harga` berdasarkan **window waktu 2 menit** dari kolom `waktu_pesan`. Gunakan sink `memory`, `queryName="agregasi_waktu"`.

In [14]:
agregasi_window = df_stream_with_total.groupBy(
    window(col("waktu_pesan"), "2 minute")
).agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

query2 = agregasi_window.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Streaming query dimulai! Status:", query2.status)

Streaming query dimulai! Status: {'message': 'Waiting for data to arrive', 'isDataAvailable': False, 'isTriggerActive': False}


**D. Menguji dengan Data Simulasi** *(bobot 25%)*

Kirim **minimal 3 batch** data (masing-masing batch berupa satu berkas CSV baru dengan jeda `time.sleep()` di antaranya) ke folder `tugas10_stream`, dengan kombinasi `nama_cabang` yang bervariasi. Setelah setiap batch, tampilkan hasil `agregasi_cabang` untuk membuktikan angka totalnya terus terakumulasi dengan benar.

In [6]:
from datetime import datetime

waktu_now = datetime.now()
pd.DataFrame({
    "pesanan_id": ["1A", "2A","3A"],
    "nama_cabang": ["paris", "swiss", "korea"],
    "nama_menu": ["matcha", "coffee", "milk"],
    "qty": [4, 5, 6],
    "harga_satuan": [99000, 86000, 54000],
    "waktu_pesan": [waktu_now.strftime("%Y-%m-%d %H:%M:%S")] *3
}).to_csv(f"{folder_tugas}/batch_1.csv",index=False)

time.sleep(6)
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()

+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|      swiss|          430000|             1|
|      paris|          396000|             1|
|      korea|          324000|             1|
+-----------+----------------+--------------+



In [7]:
waktu_now = datetime.now()
pd.DataFrame({
    "pesanan_id": ["4A", "5A","6A"],
    "nama_cabang": ["swiss", "korea", "indo"],
    "nama_menu": ["egg tart", "chewwy cookie", "bread"],
    "qty": [3, 2, 1],
    "harga_satuan": [50000, 32000, 20000],
    "waktu_pesan": [waktu_now.strftime("%Y-%m-%d %H:%M:%S")] *3
}).to_csv(f"{folder_tugas}/batch_2.csv",index=False)

time.sleep(6)
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()

+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|      swiss|          580000|             2|
|      paris|          396000|             1|
|      korea|          388000|             2|
|       indo|           20000|             1|
+-----------+----------------+--------------+



In [8]:
waktu_now = datetime.now()
pd.DataFrame({
    "pesanan_id": ["7A", "8A"],
    "nama_cabang": ["paris", "korea"],
    "nama_menu": ["americano", "kimbab"],
    "qty": [6, 1],
    "harga_satuan": [25000, 78000],
    "waktu_pesan": [waktu_now.strftime("%Y-%m-%d %H:%M:%S")] *2
}).to_csv(f"{folder_tugas}/batch_3.csv",index=False)

time.sleep(6)
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()
spark.sql("SELECT window.start, window.end, total_pendapatan, jumlah_pesanan FROM agregasi_waktu").show(truncate=False)

+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|      swiss|          580000|             2|
|      paris|          546000|             2|
|      korea|          466000|             3|
|       indo|           20000|             1|
+-----------+----------------+--------------+

+-------------------+-------------------+----------------+--------------+
|start              |end                |total_pendapatan|jumlah_pesanan|
+-------------------+-------------------+----------------+--------------+
|2026-11-09 19:06:00|2026-11-09 19:08:00|1612000         |8             |
+-------------------+-------------------+----------------+--------------+



**E. Sink Berbasis Berkas (Tantangan)** *(bobot 20%)*

Selain `memory` sink, Structured Streaming juga mendukung menulis hasil **DataFrame mentah (bukan hasil agregasi)** ke berkas secara berkelanjutan. Buat **streaming query ketiga** yang membaca dari `df_stream`, lalu menuliskannya (tanpa agregasi apa pun, cukup data mentahnya) ke folder lokal `tugas10_output` dalam format Parquet, menggunakan:
```python
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()
```
Kirim satu batch data baru, tunggu beberapa detik, lalu buktikan berkas Parquet-nya benar-benar tersimpan (`!ls tugas10_output`) dan dapat dibaca kembali sebagai DataFrame batch biasa menggunakan `spark.read.parquet()`.

In [9]:
# Buat folder dan checkpoin
shutil.rmtree("tugas10_output", ignore_errors=True)
shutil.rmtree("tugas10_checkpoint", ignore_errors=True)

# query ke 3
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

waktu_now = datetime.now()
pd.DataFrame({
    "pesanan_id": ["9A"],
    "nama_cabang": ["korea"],
    "nama_menu": ["ramen"],
    "qty": [3],
    "harga_satuan": [78000],
    "waktu_pesan": [waktu_now.strftime("%Y-%m-%d %H:%M:%S")]
}).to_csv(f"{folder_tugas}/batch_4.csv",index=False)

time.sleep(6)

spark.read.parquet("tugas10_output").show()

+----------+-----------+-------------+---+------------+-------------------+
|pesanan_id|nama_cabang|    nama_menu|qty|harga_satuan|        waktu_pesan|
+----------+-----------+-------------+---+------------+-------------------+
|        4A|      swiss|     egg tart|  3|       50000|2026-11-09 19:07:06|
|        5A|      korea|chewwy cookie|  2|       32000|2026-11-09 19:07:06|
|        6A|       indo|        bread|  1|       20000|2026-11-09 19:07:06|
|        1A|      paris|       matcha|  4|       99000|2026-11-09 19:06:55|
|        2A|      swiss|       coffee|  5|       86000|2026-11-09 19:06:55|
|        3A|      korea|         milk|  6|       54000|2026-11-09 19:06:55|
|        7A|      paris|    americano|  6|       25000|2026-11-09 19:07:15|
|        8A|      korea|       kimbab|  1|       78000|2026-11-09 19:07:15|
|        9A|      korea|        ramen|  3|       78000|2026-11-09 19:07:33|
+----------+-----------+-------------+---+------------+-------------------+



**Menghentikan semua streaming dan tutup SparkSession**

In [11]:
# Hentikan semua streaming query yang sedang berjalan
query.stop()
query2.stop()
query3.stop()

# Tutup SparkSession
spark.stop()
print("Seluruh streaming query dihentikan, SparkSession ditutup.")

Seluruh streaming query dihentikan, SparkSession ditutup.
